# r/CreepyWikipedia → Wikipedia link posts

Fetches the first page of [old.reddit.com/r/CreepyWikipedia](https://old.reddit.com/r/CreepyWikipedia/), keeps **link posts** only (`data-domain` does not start with `self.`), keeps URLs whose host matches `*.wikipedia.org`, normalizes URLs, and writes **`crawl.json`** in the **current working directory** (in Cursor/Jupyter, set the kernel’s working folder to `d:\2025_02\WR` so the file lands next to this notebook).

**robots.txt:** As of this writing, Reddit’s `https://old.reddit.com/robots.txt` has `User-agent: *` / `Disallow: /`, so a strict `can_fetch` check is **false** for the listing URL. The code **does not fetch** unless you set `ALLOW_FETCH_DESPITE_ROBOTS = True` after reading [Reddit’s Public Content Policy](https://support.reddithelp.com/hc/en-us/articles/26410290525844-Public-Content-Policy). For production use, prefer Reddit’s official API instead of HTML scraping.

Edit `USER_AGENT` in the code cell to include a real contact or project URL if you enable fetching.

In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse, urlunparse
from urllib.robotparser import RobotFileParser


import requests
from bs4 import BeautifulSoup

# --- configuration ---
LISTING_URL = "https://old.reddit.com/r/CreepyWikipedia/"
ROBOTS_URL = "https://old.reddit.com/robots.txt"
USER_AGENT = "WR-Crawl/1.0 (personal research; replace with your contact or project URL)"
REQUEST_DELAY_SEC = 1.5

# Reddit's robots.txt currently disallows "/" for User-agent: * — strict check blocks the GET.
# Set True only if you accept responsibility under Reddit's terms / public content policy.
ALLOW_FETCH_DESPITE_ROBOTS = False

OUTPUT_JSON = Path.cwd() / "crawl.json"


def normalize_wikipedia_url(url: str) -> str | None:
    """Return canonical https URL for *.wikipedia.org, or None if not a Wikipedia article URL."""
    try:
        p = urlparse(url.strip())
    except ValueError:
        return None
    if p.scheme not in ("http", "https"):
        return None
    host = (p.netloc or "").lower()
    if not (host == "wikipedia.org" or host.endswith(".wikipedia.org")):
        return None

    # Mobile: en.m.wikipedia.org → en.wikipedia.org; m.wikipedia.org → en.wikipedia.org
    parts = host.split(".")
    if len(parts) >= 4 and parts[1] == "m" and parts[-2:] == ["wikipedia", "org"]:
        host = f"{parts[0]}.wikipedia.org"
    elif host == "m.wikipedia.org":
        host = "en.wikipedia.org"
    elif host.startswith("m.") and host.endswith(".wikipedia.org"):
        host = host[2:]

    scheme = "https"
    path = p.path or "/"
    query = p.query
    frag = ""
    return urlunparse((scheme, host, path, "", query, frag))


def parse_listing_html(html: str, scraped_at: str) -> list[dict]:
    soup = BeautifulSoup(html, "html.parser")
    rows: list[dict] = []

    for div in soup.find_all("div", id=lambda x: x and str(x).startswith("thing_t3_")):
        domain = (div.get("data-domain") or "").strip()
        if domain.startswith("self."):
            continue

        raw_url = (div.get("data-url") or "").strip()
        if not raw_url.startswith(("http://", "https://")):
            continue

        wu = normalize_wikipedia_url(raw_url)
        if not wu:
            continue

        perm = (div.get("data-permalink") or "").strip()
        if perm.startswith("/"):
            reddit_permalink = "https://old.reddit.com" + perm
        elif perm.startswith("http"):
            reddit_permalink = perm
        else:
            reddit_permalink = perm

        title_el = div.select_one("p.title a.title")
        reddit_title = title_el.get_text(strip=True) if title_el else ""

        rows.append(
            {
                "wikipedia_url": wu,
                "reddit_permalink": reddit_permalink,
                "reddit_title": reddit_title,
                "scraped_at": scraped_at,
            }
        )

    return rows


def robots_allows_fetch(url: str) -> bool:
    rp = RobotFileParser()
    rp.set_url(ROBOTS_URL)
    rp.read()
    return rp.can_fetch(USER_AGENT, url)


scraped_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

allowed = robots_allows_fetch(LISTING_URL)
print("robots.txt allows fetch:", allowed)

if not allowed and not ALLOW_FETCH_DESPITE_ROBOTS:
    rows = []
    print(
        "Skipping HTTP GET (robots.txt disallows). "
        "Set ALLOW_FETCH_DESPITE_ROBOTS = True only if appropriate for your use case."
    )
else:
    if not allowed:
        print("WARNING: robots.txt disallows this URL; fetching because ALLOW_FETCH_DESPITE_ROBOTS is True.")
    time.sleep(REQUEST_DELAY_SEC)
    resp = requests.get(
        LISTING_URL,
        headers={"User-Agent": USER_AGENT},
        timeout=30,
    )
    resp.raise_for_status()
    rows = parse_listing_html(resp.text, scraped_at)

OUTPUT_JSON.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Wrote {len(rows)} records to {OUTPUT_JSON.resolve()}")
rows[:5]